# Greedy, random, temperature, top-K

The model gives you a distribution over 32,000 tokens. What you do with it is a separate design decision — with two opposite failure modes and a dial between them.

**Runs on:** GPU recommended — needs the model from notebook 02 &nbsp;·&nbsp; **Slides:** [Chapter 16 — Text Generation](../../../course-web-slides/ch16/index.html) &nbsp;·&nbsp; **Section:** 02 — Sampling strategies

---

## The naive loop, and why it is slow

In [ ]:
import keras
from keras import ops
import numpy as np
import time

mini_gpt = keras.models.load_model("mini_gpt.keras")

def generate(prompt, max_length=64):
    tokens = list(ops.convert_to_numpy(tokenizer(prompt)))
    prompt_length = len(tokens)
    for _ in range(max_length - prompt_length):
        prediction = mini_gpt(ops.convert_to_numpy([tokens]))
        prediction = ops.convert_to_numpy(prediction[0, -1])
        tokens.append(np.argmax(prediction).item())
    return tokenizer.detokenize(tokens)

prompt = "A piece of advice"
t0 = time.time()
print(generate(prompt))
print(f"\n{time.time()-t0:.1f} seconds")

**Minutes**, for 64 tokens — while training ran at 200,000 tokens per second on the same hardware.

`fit()` and `predict()` compile the per-batch computation. Calling the model directly runs the forward pass live and unoptimized at every step.

## Padding so the shape never changes

In [ ]:
def compiled_generate(prompt, max_length=64):
    tokens = list(ops.convert_to_numpy(tokenizer(prompt)))
    prompt_length = len(tokens)
    tokens = tokens + [0] * (max_length - prompt_length)
    for i in range(prompt_length, max_length):
        prediction = mini_gpt.predict(np.array([tokens]), verbose=0)
        prediction = prediction[0, i - 1]
        tokens[i] = np.argmax(prediction).item()
    return tokenizer.detokenize(tokens)

import timeit
tries = 5
compiled_generate(prompt)          # warm up: the first call compiles
t = timeit.timeit(lambda: compiled_generate(prompt), number=tries) / tries
print(f"{t:.3f} seconds per generation")

Expected output:

```
about 0.5 seconds — from minutes
```

> ⚠️ **`predict()` compiles for a **specific input shape**.** A sequence that grows by one token each step triggers recompilation every call. Padding to full length keeps the shape constant.

A large share of real-world inference cost is lost to exactly this, and it never appears as an error — only as a bill.

## The inefficiency that remains

Each call runs the model over the **whole** sequence and discards everything but one position, when the sequence changed by one token.

Attention is the only place information crosses positions. Past keys and values never change — the causal mask forbids looking ahead. **Cache them and you have the Transformer equivalent of an RNN state**: input shrinks from the whole sequence to one token, which on a long generation is a thousandfold speed-up.

Implementing it means saving and reusing intermediate arrays from every attention layer — which is exactly why you should use a library that has already done it.

## Making the strategy a parameter

In [ ]:
def compiled_generate(prompt, sample_fn, max_length=64):
    tokens = list(ops.convert_to_numpy(tokenizer(prompt)))
    prompt_length = len(tokens)
    tokens = tokens + [0] * (max_length - prompt_length)
    for i in range(prompt_length, max_length):
        prediction = mini_gpt.predict(np.array([tokens]), verbose=0)
        prediction = prediction[0, i - 1]
        next_token = ops.convert_to_numpy(sample_fn(prediction))
        tokens[i] = np.array(next_token).item()
    return tokenizer.detokenize(tokens)

def greedy_search(preds):
    return ops.argmax(preds)

print(compiled_generate(prompt, greedy_search))

**The repetition is not a bug.** The model predicts the most likely next token across a billion words on many topics; where there is no obvious continuation, guessing common words or repeated patterns is an effective strategy, and it learns that almost immediately.

Stop training very early and it would emit `"the"` forever.

## Random sampling

In [ ]:
def random_sample(preds, temperature=1.0):
    preds = preds / temperature
    return keras.random.categorical(preds[None, :], num_samples=1)[0]

print(compiled_generate(prompt, random_sample))

No longer stuck in loops — and now it **explores too much**. The output jumps around without continuity. One failure traded for its opposite.

## Temperature

In [ ]:
from functools import partial
import matplotlib.pyplot as plt

# What temperature does to a distribution, before generating anything.
logits = np.array([3.0, 2.5, 2.0, 1.0, 0.5, 0.0, -1.0, -2.0])
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for ax, T in zip(axes, [0.2, 0.5, 1.0, 2.0]):
    p = np.exp(logits / T); p /= p.sum()
    ax.bar(range(len(p)), p)
    ax.set_title(f"T = {T}   max {p.max():.2f}"); ax.set_ylim(0, 1)
plt.suptitle("Temperature acts on the LOGITS, before the softmax", y=1.04)
plt.tight_layout(); plt.show()

In [ ]:
for T in [2.0, 0.8, 0.2]:
    print(f"\n--- temperature {T} ---")
    print(compiled_generate(prompt, partial(random_sample, temperature=T)))

**T = 2.0** — subword fragments, stray identifiers, other languages. The distribution is flat enough that rare tokens win regularly, and rare tokens in a 32,000 vocabulary are mostly debris.

**T = 0.2** — converges on greedy search, repeating patterns.

Temperature is a **dial between two known failure modes**, not a fix for either.

## Top-K

In [ ]:
def top_k(preds, k=5, temperature=1.0):
    preds = preds / temperature
    top_preds, top_indices = ops.top_k(preds, k=k, sorted=False)
    choice = keras.random.categorical(top_preds[None, :], num_samples=1)[0]
    return ops.take_along_axis(top_indices, choice, axis=-1)

for k in [5, 20]:
    print(f"\n--- top-{k} ---")
    print(compiled_generate(prompt, partial(top_k, k=k)))

print("\n--- top-5, temperature 0.5 (a common production default) ---")
print(compiled_generate(prompt, partial(top_k, k=5, temperature=0.5)))

**Temperature and top-K are not the same knob.** A low temperature makes likely tokens more likely but rules **nothing** out. Top-K sets everything outside the K candidates to **zero**. They compose.

## The four, side by side

In [ ]:
print(f"{'strategy':16s} {'what it does':44s} {'failure mode'}")
print("-" * 100)
rows = [("Greedy", "argmax at every step", "repeats phrases indefinitely"),
        ("Random", "samples the full categorical distribution", "wanders, no continuity"),
        ("Temperature", "scales logits before the softmax", "a dial between the two above"),
        ("Top-K", "zeroes everything outside K candidates", "K too small collapses to greedy"),
        ("Beam search", "keeps several candidate chains alive", "expensive; can still be bland")]
for a, b, c in rows:
    print(f"{a:16s} {b:44s} {c}")

---

## What to take away

- Calling the model directly is unoptimized; `predict()` with a **constant input shape** is two orders of magnitude faster.
- Key-value caching is the remaining win, and the reason to use a serving library.
- Greedy repeats, random wanders, temperature dials between them, top-K rules tokens out.
- Repetition is the training objective working correctly, asked the wrong question.